# Biomarker S9 — M3T huấn luyện lại không rò rỉ, trích CLS 128 chiều

**Kế hoạch:** `docs/ke_hoach_m3t_cls.md`. **Cấu hình đăng ký trước:** `configs/m3t_s9.json`.
**Code:** `bsc/m3t.py` (mô hình, chuyển đổi ảnh, dấu vân tay), `bsc/m3t_train.py` (pool, train, chạy tiếp),
`bsc/ordinal.py` (bootstrap theo subject). Notebook chỉ điều phối.

**Vì sao train lại:** M3T gốc học trên 2880 subject OAI, trong đó có subject của cohort 1325 ca (44/70 subject
iMorphics nằm ở tập train của nó) → CLS của trọng số cũ đã mang nhãn KL của cohort. Ở đây M3T được train **từ đầu**
trên kho dữ liệu của nó **trừ mọi subject của cohort**, rồi mới đóng băng để trích CLS cho S8/S7.

**Thứ tự chạy — mỗi phiên Colab chạy lại mục 0 và 1 trước (bộ đệm zip mất khi ngắt phiên):**

| Phiên | Mục | Ghi chú |
|---|---|---|
| 1 | 0 → 7 | ~2–3 giờ. **Dừng ở cuối mục 7**, gửi toàn bộ output cho Claude. Mục 8 tự chặn khi chưa đăng ký. |
| 2 … n | 0, 1, 8 | Train nhiều phiên. Đặt `RUN_TRAINING = True` ở mục 8. |
| cuối | 0 → 6 (tự nạp lại file đã có), 9 → 12 | Chọn epoch, cổng cuối, trích CLS, mốc head. |

**Quy tắc:**
- Mọi sản phẩm ghi vào `.../knee_biomarkers_09_09/s9_m3t/` trên Drive, **ghi một lần** (`T.write_once_*`):
  chạy lại cell thì nạp lại file đã có, nội dung khác thì báo lỗi thay vì ghi đè.
- **Ngoại lệ duy nhất ngoài Drive** (duyệt 24/09/2026): zip đầu vào 29,5 GB được chép vào `/content/input_cache`
  làm bộ đệm đọc. Không sản phẩm nào được ghi ở đó (`assert_drive_first` chặn).
- **Trọng số M3T cũ là rò rỉ.** Chỉ dùng ở mục 2 (kiểm port), 5 (thiết kế chuyển đổi, không dùng nhãn),
  9 (so sánh), 11–12 (chỉ logit của head để đo độ lạc quan). **Không bao giờ** trích CLS làm đặc trưng từ nó —
  `load_m3t`/`extract` tự chặn khi thiếu `allow_leaky=True`.

## 0) Môi trường

In [ ]:
# ============================================================
# Moi truong: Drive + clone repo. Moi logic nam trong bsc/*.py - notebook chi goi.
# torchio PHAI la 1.2.1 nhu ban goc (knee_testing_v3.ipynb): tang cuong phu thuoc phien ban.
# ============================================================
!pip install -q torchio==1.2.1 gdown 2>/dev/null
from google.colab import drive
drive.mount("/content/drive")

REPO_URL, REPO_DIR = "https://github.com/AIVIETNAM-AIO-Tuan/bsCart-net.git", "/content/repo"
import os, sys
if not os.path.isdir(f"{REPO_DIR}/bsc"):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
for _m in [k for k in list(sys.modules) if k == "bsc" or k.startswith("bsc.")]:
    del sys.modules[_m]

import hashlib, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from bsc import io_utils as IO, m3t as M3T, m3t_train as T, ordinal as ORD, track

CFG, CFG_RAW = T.load_config(f"{REPO_DIR}/bsc/configs/m3t_s9.json")
GATES, INPUTS, EVAL = CFG_RAW["gates"], CFG_RAW["inputs"], CFG_RAW["evaluation"]
GIT_SHA = track.git_sha(REPO_DIR)
VERS = T.runtime_versions()
assert VERS["torchio"] == INPUTS["torchio_version"], (
    f"torchio {VERS['torchio']} != {INPUTS['torchio_version']}: Runtime > Restart session roi chay lai cell nay")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    _p = torch.cuda.get_device_properties(0)
    print(f"GPU {_p.name} | compute capability {_p.major}.{_p.minor} | {_p.total_memory / 2**30:.0f} GB "
          f"| TF32 {'co' if _p.major >= 8 else 'KHONG'}")
else:
    print("KHONG co GPU - muc 2, 5, 8-12 can GPU")
print(f"git {GIT_SHA[:12]} | cfg {CFG.hash()} | status {CFG_RAW['status']} | CPU {os.cpu_count()} | {VERS}")

BIOM_DIR = Path("/content/drive/MyDrive/OAI_seg/knee_biomarkers_09_09")
OUT = BIOM_DIR / "s9_m3t"                 # MOI san pham cua S9
INP = OUT / "inputs"                      # ban sao dau vao NHO (CSV nhan, trong so ro ri) tren Drive
RUN_DIR = OUT / f"run_{CFG.hash()}"       # checkpoint train, theo cfg_hash
MANIFEST = BIOM_DIR / "cohort_manifest.csv"
OUT.mkdir(parents=True, exist_ok=True)
INP.mkdir(exist_ok=True)
IO.assert_drive_first(OUT)
IO.assert_drive_first(RUN_DIR)
assert MANIFEST.exists(), f"khong thay {MANIFEST}"
print("da co trong s9_m3t/:", sorted(p.name for p in OUT.iterdir()))

## 1) Bộ đệm đầu vào, mục lục zip, nhãn M3T

Ưu tiên zip đã có trên Drive của bạn (sửa `M3T_ZIP_DRIVE` nếu để chỗ khác); không có thì tải thẳng vào bộ đệm bằng
file id đã ghim. `stage_input` kiểm kích thước và CRC của 200 npz. Mục lục zip (tên, CRC, kích thước) phải khớp bản
đã ghim trong config — khác là dữ liệu khác.

In [ ]:
# ---- zip M3T 29.5 GB -> bo dem DOC cuc bo /content/input_cache (ngoai le duy nhat ngoai Drive)
M3T_ZIP_DRIVE = Path("/content/drive/MyDrive/OAI_seg/m3t_data") / INPUTS["zip_name"]   # <<< sua neu zip o cho khac
if M3T_ZIP_DRIVE.exists():
    ZIP = T.stage_input(M3T_ZIP_DRIVE, expected_size=INPUTS["zip_size"])
else:
    _dst = Path(T.CACHE_ROOT) / INPUTS["zip_name"]
    if not (_dst.exists() and _dst.stat().st_size == INPUTS["zip_size"]):
        import gdown
        Path(T.CACHE_ROOT).mkdir(parents=True, exist_ok=True)
        print(f"khong thay {M3T_ZIP_DRIVE} -> tai bang gdown vao {T.CACHE_ROOT} (~10-30 phut)")
        try:
            gdown.download(id=INPUTS["zip_gdrive_id"], output=f"{_dst}.part", quiet=False)
        except Exception as e:
            raise RuntimeError("gdown that bai (thuong do gioi han tai cua Drive). Chep zip vao "
                               f"{M3T_ZIP_DRIVE} roi chay lai cell nay") from e
        os.replace(f"{_dst}.part", _dst)
    ZIP = T.stage_input(_dst, expected_size=INPUTS["zip_size"])

INDEX = T.npz_index(ZIP)
assert len(INDEX) == INPUTS["zip_n_npz"], f"{len(INDEX)} npz, can {INPUTS['zip_n_npz']}"
assert T.index_md5(INDEX) == INPUTS["zip_index_md5"], "noi dung zip khac ban da ghim"
T.write_once_csv(INDEX, OUT / "npz_index.csv")
READER = T.ZipNpzReader(ZIP)
MEMBER = dict(zip(INDEX.npz_name, INDEX.member))


def _mb_per_s(path, members):
    r, t0 = T.ZipNpzReader(path), time.time()
    nb = sum(r.read(m).nbytes for m in members)
    r.close()
    return nb / 1e6 / (time.time() - t0)


_speed = f"doc tu bo dem: {_mb_per_s(ZIP, INDEX.member.sample(10, random_state=0)):.0f} MB/s"
if M3T_ZIP_DRIVE.exists():
    _speed += f" | doc thang tu Drive: {_mb_per_s(M3T_ZIP_DRIVE, INDEX.member.sample(10, random_state=1)):.0f} MB/s"
print(_speed)

# ---- nhan M3T: 8149 goi, chia train/val/test THEO SUBJECT
LABELS_CSV = INP / INPUTS["labels_name"]
if not LABELS_CSV.exists():
    import gdown
    gdown.download(id=INPUTS["labels_gdrive_id"], output=str(LABELS_CSV), quiet=True)
assert hashlib.md5(LABELS_CSV.read_bytes()).hexdigest() == INPUTS["labels_md5"], "CSV nhan khac ban da ghim"
LABELS = T.attach_members(T.load_labels(LABELS_CSV), INDEX)
print(f"{len(LABELS)} goi | {LABELS.subject.nunique()} subject | {LABELS.subset.value_counts().to_dict()}")

## 2) G0 — kiểm port: trọng số rò rỉ [BEST] trên đúng 1636 gối test gốc

Bản gốc ghi **1073/1636 đúng** (acc 0,6559) và ma trận nhầm lẫn ở cell 24 của `knee_testing_v3.ipynb`.
Cổng: **1073 ± 2**. Trượt nghĩa là port, tiền xử lý hoặc trọng số không tái lập bản gốc — dừng, báo Claude.
Cục bộ đã kiểm: bản 113 epoch tái lập đúng softmax 0,703 và 0,579 ghi ở cell 28–29.

In [ ]:
LEAKY_PTH = INP / "m3t_leaky_best_model.pth"
if not LEAKY_PTH.exists():
    import gdown
    gdown.download(id=INPUTS["leaky_weights_gdrive_id"], output=str(LEAKY_PTH), quiet=True)
LEAKY, LEAKY_HASH = M3T.load_m3t(LEAKY_PTH, DEVICE, allow_leaky=True)
G0 = GATES["g0"]
assert LEAKY_HASH == G0["weights_hash"], (
    f"trong so co hash {LEAKY_HASH[:12]}, KHONG phai [BEST] {G0['weights_hash'][:12]}. {G0['weights_note']}")

G0_CSV = OUT / "g0_predictions.csv"
TEST_ORIG = LABELS[LABELS.subset == "test"].reset_index(drop=True)
assert len(TEST_ORIG) == G0["n_test"], len(TEST_ORIG)
if G0_CSV.exists():
    g0 = pd.read_csv(G0_CSV)
else:
    _dl = torch.utils.data.DataLoader(T.NpzKLDataset(TEST_ORIG, READER), batch_size=2, shuffle=False,
                                      num_workers=CFG.num_workers, pin_memory=DEVICE == "cuda")
    _ev = T.evaluate(LEAKY, _dl, DEVICE, return_outputs=True)
    g0 = pd.DataFrame(dict(npz_name=TEST_ORIG.npz_name, y=_ev["y"], pred=_ev["pred"]))
    for k in range(5):
        g0[f"logit_{k}"] = _ev["logits"][:, k]
    T.write_once_csv(g0, G0_CSV)
_cm = ORD.confusion(g0.y, g0.pred, 5)
N_OK = int(np.trace(_cm))
print(f"G0: {N_OK}/{len(g0)} dung (ban goc {G0['correct']}) | QWK {ORD.qwk(g0.y, g0.pred, 5):.3f}")
print("confusion (hang = that, cot = doan):\n", _cm)
print("tong |lech| so voi ma tran ban goc:", int(np.abs(_cm - np.array(G0["confusion"])).sum()))
T.write_once_json(dict(weights_hash=LEAKY_HASH, correct=N_OK, n=len(g0), confusion=_cm.tolist()),
                  OUT / "g0_result.json")
assert abs(N_OK - G0["correct"]) <= G0["tol"], "G0 TRUOT - DUNG, gui output cho Claude"
print("G0 DAT")

## 3) Audit cohort ↔ dữ liệu M3T

- `dess_path` cũ được sửa tiền tố (nnUNet_raw và OAI_DESS đã chuyển vào `MyDrive/OAI_seg/`).
- Mỗi ca cohort được khớp với npz của M3T: **barcode trước** (cùng lần chụp, chắc chắn), rồi subject + bên gối.
- Bảng subject × subset cho thấy **phạm vi rò rỉ** của trọng số cũ.
- Đồng thuận KL giữa hai nguồn nhãn (M3T lấy từ bộ X-quang, cohort lấy từ `KXR_SQ_BU00`) — **chỉ báo cáo**,
  dùng để diễn giải so sánh head M3T với downstream; không sửa nhãn theo kết quả này.

In [ ]:
MAN = pd.read_csv(MANIFEST).drop_duplicates("case_id").reset_index(drop=True)
_need = {"case_id", "subject", "side", "dess_path", "source_dataset", "KL"}
assert not _need - set(MAN.columns), f"manifest thieu cot {_need - set(MAN.columns)}"
assert MAN.KL.notna().all(), "manifest co ca thieu KL"
if "visit" not in MAN.columns:
    MAN["visit"] = MAN.case_id.astype(str).str.extract(r"_(V\d\d)_")[0]
print(len(MAN), "ca |", MAN.source_dataset.value_counts().to_dict())

# dess_path cu -> thu thay tien to theo thu tu (khop nguyen thanh phan duong)
PATH_REMAPS = [("/content/drive/MyDrive/nnUNet_raw", "/content/drive/MyDrive/OAI_seg/nnUNet_raw"),
               ("/content/drive/MyDrive/OAI_DESS", "/content/drive/MyDrive/OAI_seg/OAI_DESS")]   # <<< sua neu Drive khac
MAN["dess_resolved"] = [IO.resolve_path(p, PATH_REMAPS, must_exist=False) for p in MAN.dess_path.astype(str)]
MAN["img_family"] = MAN.dess_path.map(T.image_family)
print("anh tim thay theo ho file:\n", pd.crosstab(MAN.img_family, MAN.dess_resolved.notna(), margins=True))

EXPO = T.exposure_table(MAN, LABELS, INDEX).merge(
    MAN[["case_id", "KL", "dess_resolved", "img_family"]], on="case_id", validate="one_to_one")
T.write_once_csv(EXPO, OUT / "exposure.csv")
print("\nkhop npz theo ho anh:\n", pd.crosstab(EXPO.img_family, EXPO.match, margins=True))
print("\nsubject cohort o dau trong split M3T GOC (= pham vi ro ri cua trong so cu):\n",
      pd.crosstab(EXPO.source_dataset, EXPO.m3t_subset, margins=True))
print(f"\nbarcode tro goi ben kia: {int(EXPO.side_conflict.sum())} | tro subject khac: "
      f"{int(EXPO.subject_conflict.sum())} | goi co nhieu lan chup (mo ho): {int(EXPO.ambiguous.sum())}")

# dong thuan KL hai nguon: CUNG goi va CUNG lan chup (barcode, hoac V00 khop 1-1). CHI BAO CAO.
_same = EXPO.knee_in_csv & ((EXPO.match == "barcode") | (
    (EXPO.match == "subject_side") & ~EXPO.ambiguous & (EXPO.visit.astype(str) == "V00")))
_ag = EXPO[_same]
if len(_ag):
    _kc, _km = _ag.KL.astype(int), _ag.m3t_kl.astype(int)
    print(f"\nKL cohort vs KL nhan M3T: n={len(_ag)} | trung {float((_kc == _km).mean()):.3f} | "
          f"QWK {ORD.qwk(_kc, _km, 5):.3f} | cohort cao hon {float((_kc > _km).mean()):.3f}, thap hon {float((_kc < _km).mean()):.3f}")
    _xt = pd.crosstab(_kc.rename("KL_cohort"), _km.rename("KL_m3t"))
    print(_xt)
    T.write_once_csv(_xt.reset_index(), OUT / "kl_agreement.csv")
else:
    print("\nKHONG co ca nao cung goi + cung lan chup de doi chieu KL")

## 4) Pool huấn luyện — loại **mọi** subject của cohort

Loại toàn bộ subject của 1325 ca (kể cả 96 ca holdout OAI-ZIB), cả hai gối, mọi lần khám, ở cả train/val/test
của M3T. Thêm subject **thật** của npz khớp barcode (phòng khi subject trong manifest sai).
Cổng: **KL4 trong pool-train ≥ 84** (một nửa 167); trượt thì dừng để bạn chọn cách xử lý.

In [ ]:
EXCL = set(MAN.subject.map(T.norm_subject))          # ID sai => loi ngay; sua manifest truoc, khong bo qua
EXCL |= {T.parse_npz_name(n)[0] for n in EXPO.loc[EXPO.match == "barcode", "npz_name"]}
POOL = T.build_pool(LABELS, EXCL)
T.assert_subject_disjoint(POOL, EXCL, "pool vs cohort")
POOL_MD5 = T.pool_md5(POOL)
KL4_TRAIN = int(((POOL.subset == "train") & (POOL.kl_grade == 4)).sum())
print(f"loai {len(EXCL)} subject | pool {len(POOL)}/{len(LABELS)} goi, {POOL.subject.nunique()} subject | md5 {POOL_MD5[:12]}")
print("M3T goc:\n", pd.crosstab(LABELS.subset, LABELS.kl_grade, margins=True))
print("pool:\n", pd.crosstab(POOL.subset, POOL.kl_grade, margins=True))
T.write_once_csv(POOL, OUT / "pool.csv")
T.write_once_json(dict(pool_md5=POOL_MD5, n=len(POOL), n_subjects=POOL.subject.nunique(),
                       n_excluded_subjects=len(EXCL), kl4_train=KL4_TRAIN,
                       by_subset=POOL.subset.value_counts().to_dict()), OUT / "pool.json")
assert KL4_TRAIN >= GATES["pool_kl4_train_min"], (
    f"DUNG: KL4 pool-train = {KL4_TRAIN} < {GATES['pool_kl4_train_min']}. Chon: chap nhan / them npz ngoai "
    "CSV (van loai theo subject) / trong so lop - bao Claude")
print(f"KL4 pool-train {KL4_TRAIN} >= {GATES['pool_kl4_train_min']}: DAT")

## 5) Thiết kế chuyển đổi NIfTI → M3T (trọng số rò rỉ, **không dùng nhãn**)

npz của M3T sinh bằng một pipeline không rõ. Tìm phép biến đổi (hướng trục × kiểu resize) cho ảnh NIfTI của cohort
bằng tương quan voxel với npz **của chính ca đó** (cùng lần chụp):

- **Bước A — dò rộng:** 12 ca đã chắc cùng lần chụp, cả 48 hướng × 5 kiểu resize → xác nhận trục lát cắt 0,70 mm
  rơi vào trục nào của M3T (kế hoạch giả định trục 0).
- **Bước B — theo họ file × bên gối:** ~300 ca, 16 hướng hợp lệ × 5 kiểu resize. OAI-ZIB (`D001`): bên gối trong
  manifest chưa kiểm (S1 gán cứng `R`) → so với **cả hai** npz của subject, ảnh tự chọn bên (`pick_best_tag`).
- **Cổng G4.1 (ảnh), từng họ × bên:** r trung vị ≥ 0,99, phân vị 1% ≥ 0,97, hơn spec tốt nhất có **hướng khác**
  ≥ 0,1, hệ số góc 0,95–1,05. **G4.2/G4.3 (CLS, head)** với trọng số rò rỉ ở đây, làm lại với trọng số mới ở mục 10.

In [ ]:
# cap CUNG lan chup: barcode (chac chan) hoac subject+ben khop 1-1 o V00 (npz M3T gan nhu chi V00)
EXPO["same_acq"] = (EXPO.match == "barcode") | (
    (EXPO.match == "subject_side") & ~EXPO.ambiguous & (EXPO.visit.astype(str) == "V00"))
# ben goi THAT cua anh: barcode chi dung goi (ke ca khi manifest ghi nham ben), con lai theo manifest
EXPO["side_img"] = [T.parse_npz_name(n)[2] if m == "barcode" else s
                    for n, m, s in zip(EXPO.npz_name, EXPO.match, EXPO.side)]
KNEE_NPZ = INDEX.groupby(["subject", "side"]).npz_name.apply(list).to_dict()
SIDE_UNVERIFIED = {"D001_oaizib"}                    # S1 gan cung side=R, visit chua kiem


def _candidates(r):
    """[(npz_name, tag)] de doi chieu voi anh cua ca r; tag = ben goi cua npz."""
    if r.img_family in SIDE_UNVERIFIED:
        return [(KNEE_NPZ[(r.subject, s)][0], s) for s in "LR"
                if r.subject is not None and len(KNEE_NPZ.get((r.subject, s), [])) == 1]
    return [(r.npz_name, r.side_img)] if r.same_acq and isinstance(r.npz_name, str) else []


EXPO["cands"] = [_candidates(r) for r in EXPO.itertuples()]
DESIGNABLE = EXPO[EXPO.dess_resolved.notna() & (EXPO.cands.str.len() > 0)]
print("ca thiet ke duoc (co anh + npz doi chieu) theo ho:", DESIGNABLE.img_family.value_counts().to_dict())


def _pairs(rows, desc):
    """(case_id, NIfTI [Z,Y,X], spacing, npz, tag) - doc TUNG ca, khong giu khoi NIfTI trong RAM."""
    for r in tqdm(list(rows.itertuples()), desc=desc):
        arr, sp = IO.load_nii(r.dess_resolved)
        for name, tag in r.cands:
            yield r.case_id, arr, sp, READER.read(MEMBER[name]), tag


# ---- Buoc A: 48 huong x 5 kieu resize tren ca chac cung lan chup (khong phai D001)
A_CSV = OUT / "conversion_search_broad.csv"
if A_CSV.exists():
    SEARCH_A = pd.read_csv(A_CSV)
else:
    _a = DESIGNABLE[~DESIGNABLE.img_family.isin(SIDE_UNVERIFIED)]
    assert len(_a), "khong co ca nao chac cung lan chup de do rong - bao Claude"
    _k = GATES["design_broad_n_cases"] // 2
    _rows = pd.concat([g.sample(min(len(g), _k), random_state=0) for _, g in _a.groupby("side")])
    SEARCH_A = M3T.search_conversion(_pairs(_rows, "buoc A"), orients=list(M3T.ORIENTATIONS))
    T.write_once_csv(SEARCH_A, A_CSV)
SA = M3T.summarize_conversion(SEARCH_A)
print(SA.head(8).to_string())
assert SEARCH_A.src_axis.nunique() == 1, f"truc lat cat khac nhau giua cac ca: {SEARCH_A.src_axis.unique()}"
SRC_AXIS = int(SEARCH_A.src_axis.iloc[0])
DST_AXIS = M3T.dst_axis_of(SA.iloc[0].orient, SRC_AXIS)
print(f"\ntruc lat cat 0.70 mm cua NIfTI = {SRC_AXIS} -> truc {DST_AXIS} cua M3T (ke hoach gia dinh 0)")
if DST_AXIS != 0:
    print("CANH BAO: truc lat cat KHONG ve truc 0 - buoc B do 16 huong dua truc lat cat ve truc", DST_AXIS)

In [ ]:
# ---- Buoc B: 16 huong x 5 kieu resize, ~300 ca chia deu theo ho file; D001 so voi ca hai ben
B_CSV = OUT / "conversion_search.csv"
if B_CSV.exists():
    SEARCH_B = pd.read_csv(B_CSV)
else:
    _k = int(np.ceil(GATES["design_n_cases"] / DESIGNABLE.img_family.nunique()))
    _rows = pd.concat([g.sample(min(len(g), _k), random_state=0) for _, g in DESIGNABLE.groupby("img_family")])
    SEARCH_B = M3T.search_conversion(_pairs(_rows, "buoc B"), dst_axis=DST_AXIS)
    T.write_once_csv(SEARCH_B, B_CSV)
SEARCH_B = SEARCH_B.merge(EXPO[["case_id", "img_family"]], on="case_id", validate="many_to_one")
_unv = SEARCH_B.img_family.isin(SIDE_UNVERIFIED)
_parts = [SEARCH_B[~_unv]]
if _unv.any():
    _keep, SIDE_DESIGN = M3T.pick_best_tag(SEARCH_B[_unv])
    _parts.append(_keep)
    T.write_once_csv(SIDE_DESIGN, OUT / "side_verdict_design.csv")
    _sv = SIDE_DESIGN.merge(EXPO[["case_id", "side"]], on="case_id")
    print(f"D001: ben theo anh khac ben manifest o {int((_sv.best_tag != _sv.side).sum())}/{len(_sv)} ca | "
          f"r ben tot trung vi {_sv.r_best.median():.3f}, ben kia {_sv.r_other.median():.3f}")
SEARCH_B = pd.concat(_parts, ignore_index=True)

SPEC_ROWS = []
for (_fam, _tag), _g in SEARCH_B.groupby(["img_family", "tag"]):
    _gate = M3T.image_gate(M3T.summarize_conversion(_g), GATES["image"])
    SPEC_ROWS.append(dict(img_family=_fam, side=_tag, **{k: v for k, v in _gate.items() if k != "checks"},
                          **{f"ok_{k}": v for k, v in _gate["checks"].items()}))
SPECS = pd.DataFrame(SPEC_ROWS)
print(SPECS.to_string())

In [ ]:
# ---- G4.2 / G4.3 voi trong so RO RI (chi so CLS/head cua anh chuyen doi vs npz cung ca; khong luu CLS)
GATE_LEAKY_JSON = OUT / "conversion_gate_leaky.json"
_cands_of = EXPO.set_index("case_id")


def _design_ids(fam, side):
    return sorted(SEARCH_B[(SEARCH_B.img_family == fam) & (SEARCH_B.tag == side)].case_id.unique())


def _cls_gate(model, fam, side, spec, allow_leaky):
    ids = _design_ids(fam, side)
    if len(ids) < 2:
        return dict(passed=False, n=len(ids), reason="qua it ca thiet ke (< 2) - khong kiem duoc")
    npz = {c: dict((t, n) for n, t in _cands_of.loc[c, "cands"])[side] for c in ids}
    conv = (M3T.convert_case(_cands_of.loc[c, "dess_resolved"], spec, DST_AXIS)
            for c in tqdm(ids, desc=f"{fam}|{side}"))
    ca, la = M3T.extract(model, conv, DEVICE, allow_leaky=allow_leaky)
    cb, lb = M3T.extract(model, (READER.read(MEMBER[npz[c]]) for c in ids), DEVICE, allow_leaky=allow_leaky)
    return M3T.conversion_gate(ca, cb, la, lb, GATES["cls"])


if GATE_LEAKY_JSON.exists():
    GATE_LEAKY = json.load(open(GATE_LEAKY_JSON))
else:
    GATE_LEAKY = {f"{r.img_family}|{r.side}": dict(spec=r.spec, **_cls_gate(LEAKY, r.img_family, r.side, r.spec, True))
                  for r in SPECS.itertuples()}
    T.write_once_json(GATE_LEAKY, GATE_LEAKY_JSON)

CONV_JSON = OUT / "conversion_specs.json"
if not CONV_JSON.exists():
    T.write_once_json(dict(src_axis=SRC_AXIS, dst_axis=DST_AXIS, gates_image=GATES["image"], gates_cls=GATES["cls"],
                           groups={f"{r.img_family}|{r.side}": dict(
                               spec=r.spec, image_pass=r.passed, r_median=r.r_median, r_p01=r.r_p01, margin=r.margin,
                               slope_median=r.slope_median, n=r.n,
                               cls_pass_leaky=GATE_LEAKY[f"{r.img_family}|{r.side}"]["passed"])
                               for r in SPECS.itertuples()}), CONV_JSON)
CONV = json.load(open(CONV_JSON))
DST_AXIS = CONV["dst_axis"]
print(pd.DataFrame(CONV["groups"]).T.to_string())
print(pd.DataFrame({k: {m: v[m] for m in ("nn_self", "dist_ratio", "head_agree", "d_ekl", "passed")}
                    for k, v in GATE_LEAKY.items()}).T.to_string())

## 6) Kiểm trùng ảnh — dấu vân tay

Lưới an toàn cho lỗi ID (subject sai, bên gối sai của OAI-ZIB): mọi ca cohort được chuyển đổi rồi so dấu vân tay với
**mọi npz trong pool**. Cổng: **không ca nào** có tương quan ≥ `fingerprint_hit_r` = 0,90 với npz của pool; trúng thì
dừng để xem tương quan voxel, không tự loại. Kiểm dương: ca có npz của chính nó phải tự khớp. OAI-ZIB: bên gối được xác
minh ở đây cho **mọi** ca có npz. (~1325 ảnh NIfTI đọc từ Drive, ~40–60 phút.)

Hiệu chỉnh ngưỡng trên npz thật (25/09/2026): khác subject — max với 3000 npz trung vị 0,796, cao nhất 0,852;
cùng gối **khác** lần chụp — trung vị 0,957; cùng subject gối kia — cao nhất 0,715.

In [ ]:
FP_CSV, FP_JSON = OUT / "fingerprint_check.csv", OUT / "fingerprint_summary.json"
HIT_R = GATES["fingerprint_hit_r"]
GROUP_SPEC = {k: v["spec"] for k, v in CONV["groups"].items() if v["image_pass"]}
if FP_CSV.exists():
    FPC = pd.read_csv(FP_CSV)
else:
    BANK = T.fingerprint_bank(INDEX.member, READER, num_workers=CFG.num_workers)
    _in_pool = INDEX.npz_name.isin(POOL.npz_name).to_numpy()
    POOL_BANK, POOL_NAMES = BANK[_in_pool], INDEX.npz_name[_in_pool].to_numpy()
    ROW_OF = {n: i for i, n in enumerate(INDEX.npz_name)}
    _recs = []
    for r in tqdm(list(EXPO.itertuples()), desc="dau van tay cohort"):
        rec = dict(case_id=r.case_id, img_family=r.img_family, side_manifest=r.side, side_verified=None,
                   r_self=np.nan, r_pool=np.nan, pool_npz=None, status="ok")
        sides = ["L", "R"] if r.img_family in SIDE_UNVERIFIED else [r.side_img]
        specs = {s: GROUP_SPEC.get(f"{r.img_family}|{s}") for s in sides}
        if r.dess_resolved is None:
            rec["status"] = "no_image"
        elif not any(specs.values()):
            rec["status"] = "no_spec"
        else:
            arr, sp = IO.load_nii(r.dess_resolved)
            res = []
            for s, spec in specs.items():
                if spec is None:
                    continue
                fp = M3T.fingerprint(M3T.nifti_to_m3t(arr, sp, spec, dst_axis=DST_AXIS))
                own = KNEE_NPZ.get((r.subject, s), [])
                r_self = max((float(BANK[ROW_OF[n]] @ fp) for n in own), default=np.nan)
                res.append((s, fp, r_self))
            scored = [x for x in res if not np.isnan(x[2])]
            s_best, _, r_self = max(scored, key=lambda x: x[2]) if scored else (None, None, np.nan)
            pool_hits = [M3T.max_corr(fp, POOL_BANK) for _, fp, _ in res]
            rp, ip = max(pool_hits)
            rec.update(side_verified=s_best if (s_best is not None and r_self >= HIT_R) else None,
                       r_self=r_self, r_pool=rp, pool_npz=POOL_NAMES[ip])
        _recs.append(rec)
    FPC = pd.DataFrame(_recs)
    T.write_once_csv(FPC, FP_CSV)

_ctrl = FPC.merge(EXPO[["case_id", "same_acq"]], on="case_id")
_ctrl = _ctrl[_ctrl.same_acq & _ctrl.r_self.notna() & ~_ctrl.img_family.isin(SIDE_UNVERIFIED)]
_d001 = FPC[FPC.img_family.isin(SIDE_UNVERIFIED) & (FPC.status == "ok")]
HITS = FPC[FPC.r_pool >= HIT_R]
FP_SUM = dict(n=len(FPC), status=FPC.status.value_counts().to_dict(), hit_r=HIT_R,
              positive_control_n=len(_ctrl), positive_control_rate=float((_ctrl.r_self >= HIT_R).mean()) if len(_ctrl) else None,
              d001_n=len(_d001), d001_side_verified=int(_d001.side_verified.notna().sum()),
              d001_side_differs=int((_d001.side_verified.notna() & (_d001.side_verified != _d001.side_manifest)).sum()),
              r_pool_max=float(FPC.r_pool.max()), n_pool_hits=len(HITS), pool_hits=HITS.case_id.tolist())
if not FP_JSON.exists():
    T.write_once_json(FP_SUM, FP_JSON)
print(json.dumps(FP_SUM, indent=1))
print("10 ca giong pool nhat:\n", FPC.nlargest(10, "r_pool")[["case_id", "img_family", "r_pool", "pool_npz", "r_self"]])
assert len(HITS) == 0, f"DUNG: {len(HITS)} ca cohort TRUNG anh voi npz trong pool (ro ri qua ID sai) - bao Claude"
print("khong ca nao trung pool: DAT")

## 7) Pilot bootstrap + bản nháp đăng ký trước — **DỪNG Ở ĐÂY**

**Pilot:** trên dự đoán OOF cũ của B (`s7_ordinal/oof_predictions_seed0.csv`), so `s6_all` với
`s6_all_plus_radiomics` bằng đúng giao thức mục 6 (bootstrap theo subject, 10.000 lượt, seed 0, CI 95%).
Đây là kiểm **độ chính xác đạt được**, **một seed** vì artifact cũ chỉ lưu seed 0; **không** dùng độ rộng CI để
định nghĩa δ (δ = 0,02 đã ghim).

**Đăng ký:** cell ghi `preregistration_draft.json`. Gửi output mục 0–7 cho Claude → Claude commit `status = REGISTERED`
cùng mục Event.md → `git pull` (chạy lại mục 0) rồi mới sang mục 8. Mục 8 tự chặn khi chưa đăng ký.

In [ ]:
PILOT_JSON = OUT / "pilot_bootstrap.json"
if PILOT_JSON.exists():
    PILOT = json.load(open(PILOT_JSON))
else:
    _P, _BS = EVAL["pilot"], EVAL["bootstrap"]
    _oof = pd.read_csv(BIOM_DIR / _P["source"])
    _ca, _cb = f"pred_{_P['a']}__{_P['model']}", f"pred_{_P['b']}__{_P['model']}"
    assert {_ca, _cb, "KL", "subject"} <= set(_oof.columns), f"thieu cot {_ca} / {_cb}"
    assert _oof[[_ca, _cb, "KL", "subject"]].notna().all().all(), "OOF co ca thieu du doan"
    _cls = sorted(_oof.KL.astype(int).unique())
    _ix = {c: i for i, c in enumerate(_cls)}
    _y = _oof.KL.astype(int).map(_ix).to_numpy()
    _a, _b = (_oof[c].astype(int).map(_ix).to_numpy() for c in (_ca, _cb))
    _t0 = time.time()
    _r = ORD.bootstrap_delta(_y, _a, _b, "qwk", len(_cls), n_boot=_BS["n_boot"], seed=_BS["seed"],
                             alpha=_BS["alpha"], groups=_oof.subject.astype(str).to_numpy())
    PILOT = dict(_r, ci_width=_r["ci_high"] - _r["ci_low"], n=len(_oof), n_subjects=_oof.subject.nunique(),
                 seeds=[0], model=_P["model"], a=_P["a"], b=_P["b"], qwk_a=ORD.qwk(_y, _a, len(_cls)),
                 qwk_b=ORD.qwk(_y, _b, len(_cls)), seconds=round(time.time() - _t0, 1), note=_P["note"])
    T.write_once_json(PILOT, PILOT_JSON)
print(json.dumps(PILOT, indent=1))

PREREG_JSON = OUT / "preregistration_draft.json"
if not PREREG_JSON.exists():
    T.write_once_json(dict(git=GIT_SHA, cfg_hash=CFG.hash(), config=CFG_RAW, versions=VERS,
                           pool=json.load(open(OUT / "pool.json")), g0=json.load(open(OUT / "g0_result.json")),
                           conversion=CONV, conversion_gate_leaky=GATE_LEAKY, fingerprint=FP_SUM, pilot=PILOT,
                           leaky_hashes=sorted(M3T.LEAKY_HASHES | {LEAKY_HASH})), PREREG_JSON)
print("\n" + "=" * 90 + "\nDUNG O DAY. Gui toan bo output muc 0-7 cho Claude. Sau commit dang ky (status = REGISTERED)"
      "\nchay lai muc 0 (git pull), muc 1, roi muc 8.\n" + "=" * 90)

## 8) Train — chỉ khi `status = REGISTERED`

Mỗi phiên: chạy mục 0, 1 rồi cell dưới. Lần đầu đo **benchmark 100 bước** (thay số ước bằng số đo trước khi cam kết
GPU). `RUN_TRAINING = True` để train; `fit` tự chạy tiếp từ `last.pt`, dừng êm trước khi hết `SESSION_HOURS`,
khi gặp file `STOP` trong thư mục run, hoặc khi quy tắc dừng sớm thỏa (không cải thiện 0,005 trên điểm trượt trong
30 epoch; tối thiểu 60, tối đa 300). Cổng: **epoch 30, val QWK trượt ≥ 0,5**.

In [ ]:
assert CFG_RAW["status"] == "REGISTERED", (
    f"DUNG: cau hinh chua dang ky truoc (status = {CFG_RAW['status']}). Gui output muc 0-7 cho Claude; "
    "sau commit dang ky, chay lai muc 0 (git pull), muc 1, roi toi day.")
POOL = pd.read_csv(OUT / "pool.csv", dtype={"subject": str})
POOL_MD5 = T.pool_md5(POOL)
assert POOL_MD5 == json.load(open(OUT / "pool.json"))["pool_md5"], "pool.csv khac pool.json"
TRAIN_DS = T.NpzKLDataset(POOL[POOL.subset == "train"], READER, transform=T.get_augment(CFG.augment))
VAL_DS = T.NpzKLDataset(POOL[POOL.subset == "val"], READER)
print(f"train {len(TRAIN_DS)} | val {len(VAL_DS)} | run {RUN_DIR.name}")
RUN_DIR.mkdir(exist_ok=True)
BENCH_JSON = RUN_DIR / "benchmark.json"
if not BENCH_JSON.exists():
    torch.manual_seed(CFG.seed)
    _bm = T.benchmark(M3T.build_m3t(**CFG.model), TRAIN_DS, VAL_DS, CFG, DEVICE, n_steps=100)
    T.write_once_json(dict(_bm, gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
                           versions=VERS), BENCH_JSON)
BENCH = json.load(open(BENCH_JSON))
print(json.dumps(BENCH, indent=1))
print(f"uoc {BENCH['est_epoch_min']:.1f} phut/epoch -> {BENCH['est_hours_min_epochs']:.1f} gio (toi thieu "
      f"{CFG.min_epochs} epoch) .. {BENCH['est_hours_max_epochs']:.1f} gio (toi da {CFG.max_epochs})"
      + (" | NUT THAT O CPU (doc + tang cuong)" if BENCH["cpu_bound"] else ""))

RUN_TRAINING = False      # <<< True de train trong phien nay (tu chay tiep tu checkpoint)
SESSION_HOURS = 11.0      # <<< dung em truoc khi het phien Colab
if RUN_TRAINING:
    torch.manual_seed(CFG.seed)                      # khoi tao tat dinh; bi ghi de khi chay tiep
    T.fit(M3T.build_m3t(**CFG.model), TRAIN_DS, VAL_DS, RUN_DIR, CFG, POOL_MD5, DEVICE,
          deadline=time.time() + SESSION_HOURS * 3600)

In [ ]:
import matplotlib.pyplot as plt
_state, _src = T.load_checkpoint(RUN_DIR)
HIST = _state["history"] if _state else []
print(f"{len(HIST)} epoch da xong ({_src}) | dung? {T.should_stop(HIST, CFG) if HIST else '-'}")
if HIST:
    _q = [h["val_qwk"] for h in HIST]
    _sm = T.trailing_means(_q, CFG.window)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(_q, lw=0.8, label="val QWK")
    ax[0].plot([np.nan if v is None else v for v in _sm], lw=2, label=f"trung binh truot {CFG.window}")
    ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot([h["train_loss"] for h in HIST], label="train"); ax[1].plot([h["val_loss"] for h in HIST], label="val")
    ax[1].set_xlabel("epoch"); ax[1].set_title("loss"); ax[1].legend()
    plt.tight_layout(); plt.show()
    _e30 = GATES["epoch30"]
    if len(HIST) > _e30["epoch"]:
        _v = _sm[_e30["epoch"]]
        assert _v >= _e30["min_smoothed_val_qwk"], f"DUNG: epoch 30 val QWK truot {_v:.3f} < {_e30['min_smoothed_val_qwk']} - bao Claude"
        print(f"cong epoch 30: {_v:.3f} >= {_e30['min_smoothed_val_qwk']} DAT")

## 9) Chọn epoch + sanity so với M3T rò rỉ

Cửa sổ **trailing 5 epoch** có trung bình val QWK cao nhất (hòa → epoch sớm nhất); checkpoint chính là epoch cuối
cửa sổ. Chỉ đọc val QWK — không một điểm S7/S8 nào ảnh hưởng. **Sanity:** M3T mới và M3T rò rỉ trên cùng tập test
M3T đã thu hẹp (bỏ subject cohort): lệch acc và QWK đều ≤ 0,05. Phiên cuối: chạy mục 0–6 (tự nạp lại) trước mục này.

In [ ]:
_state, _ = T.load_checkpoint(RUN_DIR)
HIST = _state["history"]
_stop, _why = T.should_stop(HIST, CFG)
assert _stop, "chua xong train - quay lai muc 8"
SEL = T.select_epoch(HIST, CFG.window)
T.write_once_json(dict(SEL, reason=_why, n_epochs=len(HIST), cfg_hash=CFG.hash(), pool_md5=POOL_MD5),
                  RUN_DIR / "selection.json")
print(SEL, "|", _why)
KNOWN_LEAKY = set(M3T.LEAKY_HASHES) | {json.load(open(OUT / "g0_result.json"))["weights_hash"]}
NEW, NEW_HASH = M3T.load_m3t(RUN_DIR / f"epoch_{SEL['epoch']:03d}.pt", DEVICE, **CFG.model)
assert NEW_HASH not in KNOWN_LEAKY, "trong so chon ra trung trong so ro ri"

SANITY_JSON = RUN_DIR / "sanity.json"
if not SANITY_JSON.exists():
    _test = POOL[POOL.subset == "test"]
    _dl = torch.utils.data.DataLoader(T.NpzKLDataset(_test, READER), batch_size=2, shuffle=False,
                                      num_workers=CFG.num_workers)
    _new, _old = T.evaluate(NEW, _dl, DEVICE), T.evaluate(LEAKY, _dl, DEVICE)
    T.write_once_json(dict(n=len(_test), new=_new, leaky=_old, d_acc=_new["acc"] - _old["acc"],
                           d_qwk=_new["qwk"] - _old["qwk"], weights_hash=NEW_HASH), SANITY_JSON)
SANITY = json.load(open(SANITY_JSON))
print(f"test thu hep n={SANITY['n']}: moi acc {SANITY['new']['acc']:.3f} QWK {SANITY['new']['qwk']:.3f} | "
      f"ro ri acc {SANITY['leaky']['acc']:.3f} QWK {SANITY['leaky']['qwk']:.3f}")
_sg = GATES["sanity"]
assert abs(SANITY["d_acc"]) <= _sg["max_abs_d_acc"] and abs(SANITY["d_qwk"]) <= _sg["max_abs_d_qwk"], \
    "SANITY TRUOT - bao Claude"
print("sanity DAT")

## 10) Cổng chuyển đổi cuối với trọng số mới → chọn đường trích

G4.2/G4.3 làm lại trên đúng các ca thiết kế của mục 5, bằng checkpoint chính. **Qua ở mọi họ × bên và mọi ca có
spec** → cả 1325 ca đi **một** đường (NIfTI chuyển đổi). **Trượt** → chỉ ca có npz cùng lần chụp, mọi feature set
(kể cả radiomics) ở S8/S7 phải chạy trên cùng giao đó.

In [ ]:
FINAL_JSON = RUN_DIR / "conversion_gate_final.json"
if FINAL_JSON.exists():
    GATE_FINAL = json.load(open(FINAL_JSON))
else:
    GATE_FINAL = {}
    for _key, _g in CONV["groups"].items():
        _fam, _side = _key.split("|")
        GATE_FINAL[_key] = (dict(spec=_g["spec"], **_cls_gate(NEW, _fam, _side, _g["spec"], False))
                            if _g["image_pass"] else dict(spec=_g["spec"], passed=False, reason="G4.1 anh truot o muc 5"))
    T.write_once_json(GATE_FINAL, FINAL_JSON)
print(pd.DataFrame(GATE_FINAL).T.drop(columns=["checks"], errors="ignore").to_string())

OK_SPEC = {k: v["spec"] for k, v in GATE_FINAL.items() if v["passed"]}
_fp = FPC.set_index("case_id")


def spec_for(r):
    """Spec cho ca r, hoac None neu khong chuyen doi chac chan duoc."""
    if r.dess_resolved is None:
        return None
    if r.img_family in SIDE_UNVERIFIED:
        side = _fp.loc[r.case_id, "side_verified"]
        if isinstance(side, str):
            return OK_SPEC.get(f"{r.img_family}|{side}")
        both = {OK_SPEC.get(f"{r.img_family}|{s}") for s in "LR"}
        return both.pop() if len(both) == 1 and None not in both else None
    return OK_SPEC.get(f"{r.img_family}|{r.side_img}")


def npz_for(r):
    """npz CUNG lan chup cua ca r cho duong npz_intersection, hoac None."""
    if r.img_family in SIDE_UNVERIFIED:
        side = _fp.loc[r.case_id, "side_verified"]
        names = KNEE_NPZ.get((r.subject, side), []) if isinstance(side, str) else []
        return names[0] if len(names) == 1 else None
    return r.npz_name if r.same_acq and isinstance(r.npz_name, str) else None


EXPO["spec"] = [spec_for(r) for r in EXPO.itertuples()]
EXPO["npz_use"] = [npz_for(r) for r in EXPO.itertuples()]
_uncovered = EXPO[EXPO.spec.isna()]
PATH = "nifti_all" if len(_uncovered) == 0 else "npz_intersection"
PLAN_JSON = RUN_DIR / "extraction_plan.json"
if not PLAN_JSON.exists():
    T.write_once_json(dict(path=PATH, n_cohort=len(EXPO), n_uncovered=len(_uncovered),
                           uncovered_by_family=_uncovered.img_family.value_counts().to_dict(),
                           uncovered=_uncovered.case_id.tolist(), n_npz_same_acq=int(EXPO.npz_use.notna().sum())),
                      PLAN_JSON)
PLAN = json.load(open(PLAN_JSON))
PATH = PLAN["path"]
print(f"duong trich: {PATH} | ca khong chuyen doi chac chan duoc: {PLAN['n_uncovered']} {PLAN['uncovered_by_family']}"
      f" | ca co npz cung lan chup: {PLAN['n_npz_same_acq']}")

## 11) Trích CLS — checkpoint chính + 4 checkpoint còn lại của cửa sổ (độ nhạy)

`m3t_cls.csv` = checkpoint chính; `m3t_cls_eXXX.csv` = từng checkpoint trong cửa sổ đã ghim (chỉ để chạy lại cặp
B `__nosel` chính). Logit head của M3T rò rỉ lưu riêng `m3t_leaky_head.csv` (**không có CLS**) chỉ để đo độ lạc quan
ở mục 12. Cổng: không NaN, `case_id` duy nhất, không chiều hằng, hash ∉ trọng số rò rỉ, đủ số ca của đường trích.

In [ ]:
CLS_CSV = RUN_DIR / "m3t_cls.csv"
WIN = SEL["window"]
MODELS = {e: M3T.load_m3t(RUN_DIR / f"epoch_{e:03d}.pt", DEVICE, **CFG.model)[0] for e in WIN}
HASHES = {e: m.weights_hash for e, m in MODELS.items()}
assert not set(HASHES.values()) & KNOWN_LEAKY, "checkpoint trong cua so trung trong so ro ri"
if PATH == "nifti_all":
    ROWS = EXPO.sort_values("case_id")
    _vol = lambda r: M3T.convert_case(r.dess_resolved, r.spec, DST_AXIS)
else:
    ROWS = EXPO[EXPO.npz_use.notna()].sort_values("case_id")
    _vol = lambda r: READER.read(MEMBER[r.npz_use])
print(f"{PATH}: {len(ROWS)} ca | KL {ROWS.KL.astype(int).value_counts().sort_index().to_dict()}")

if not CLS_CSV.exists():
    _feat = {e: [] for e in WIN}
    _log = {e: [] for e in WIN}
    _leaky = []
    for r in tqdm(list(ROWS.itertuples()), desc="trich CLS"):
        v = _vol(r)
        for e, m in MODELS.items():
            c, l = M3T.extract(m, [v], DEVICE, bs=1)
            _feat[e].append(c[0])
            _log[e].append(l[0])
        _leaky.append(M3T.extract(LEAKY, [v], DEVICE, bs=1, allow_leaky=True)[1][0])
    _input = "nifti" if PATH == "nifti_all" else "npz"
    for e in WIN:
        _df = M3T.cls_frame(ROWS.case_id, np.stack(_feat[e]), np.stack(_log[e]),
                            dict(weights_hash=HASHES[e], epoch=e, input=_input, cfg_hash=CFG.hash(), pool_md5=POOL_MD5))
        T.write_once_csv(_df, RUN_DIR / f"m3t_cls_e{e:03d}.csv")
        if e == SEL["epoch"]:
            T.write_once_csv(_df, CLS_CSV)
    _lk = pd.DataFrame(np.stack(_leaky), columns=[f"leakylogit_{k}" for k in range(5)])
    _lk.insert(0, "case_id", ROWS.case_id.to_numpy())
    T.write_once_csv(_lk, RUN_DIR / "m3t_leaky_head.csv")
    T.write_once_json(dict(path=PATH, n=len(ROWS), main_epoch=SEL["epoch"], window=WIN,
                           weights_hash={str(e): h for e, h in HASHES.items()}, cfg_hash=CFG.hash(), pool_md5=POOL_MD5,
                           dst_axis=DST_AXIS, specs=OK_SPEC, git=GIT_SHA, versions=VERS,
                           kl_counts=ROWS.KL.astype(int).value_counts().sort_index().to_dict(),
                           known_leaky=sorted(KNOWN_LEAKY)), RUN_DIR / "m3t_cls_meta.json")

CLS = pd.read_csv(CLS_CSV)
_fc = [c for c in CLS.columns if M3T.is_cls_col(c)]
assert len(_fc) == {**M3T.M3T_CFG, **CFG.model}["emb_dim"] and len(CLS) == len(ROWS) and CLS.case_id.is_unique
assert np.isfinite(CLS[_fc].to_numpy()).all() and (CLS[_fc].std() > 0).all()
assert set(CLS.prov_weights_hash) == {HASHES[SEL["epoch"]]} and not set(CLS.prov_weights_hash) & KNOWN_LEAKY
print(f"m3t_cls.csv: {len(CLS)} ca x {len(_fc)} chieu | epoch {SEL['epoch']} | hash {HASHES[SEL['epoch']][:12]} | DAT")

## 12) Mốc trực tiếp: head M3T đóng băng

Dự đoán của chính head M3T (argmax logit) so với nhãn cohort, trên 246 ca test S8 (`s8_holdout_v2/holdout_split.csv`)
và trên cả 1229 ca. Lưu ý diễn giải: head học KL từ bộ X-quang của M3T, cohort dùng `KXR_SQ_BU00` (xem mục 3).
Tùy chọn: head **rò rỉ** tách theo phơi nhiễm (subject nằm trong tập train M3T gốc hay không) để đo độ lạc quan.

In [ ]:
HEAD_JSON = RUN_DIR / "head_baseline.json"
_lg = CLS[[c for c in CLS.columns if M3T.is_logit_col(c)]].to_numpy()
_head = pd.DataFrame(dict(case_id=CLS.case_id, head_pred=_lg.argmax(1)))
_v2 = pd.read_csv(BIOM_DIR / "s6_fcl" / "biomarker_table_v2.csv", usecols=["case_id", "KL"])
_split = pd.read_csv(BIOM_DIR / "s8_holdout_v2" / "holdout_split.csv")


def _metrics(d, pred_col):
    y, p = d.KL.astype(int).to_numpy(), d[pred_col].astype(int).to_numpy()
    prf = ORD.per_class_prf(y, p, 5)
    return dict(n=len(d), qwk=ORD.qwk(y, p, 5), acc=float((y == p).mean()), mae=ORD.mae(y, p),
                off_by_2=ORD.off_by_rate(y, p, 2), recall=prf["recall"].round(4).tolist(),
                precision=prf["precision"].round(4).tolist(), confusion=ORD.confusion(y, p, 5).tolist())


_all = _v2.merge(_head, on="case_id", how="inner")
_s8 = _split[_split.split == "test"][["case_id"]].merge(_all, on="case_id", how="inner")
HEAD = dict(all_1229=_metrics(_all, "head_pred"), s8_test=_metrics(_s8, "head_pred"),
            coverage=dict(all=f"{len(_all)}/{len(_v2)}", s8_test=f"{len(_s8)}/{int((_split.split == 'test').sum())}"))
_lk = pd.read_csv(RUN_DIR / "m3t_leaky_head.csv")
_lk["leaky_pred"] = _lk[[f"leakylogit_{k}" for k in range(5)]].to_numpy().argmax(1)
_lk = _lk.merge(_v2, on="case_id").merge(EXPO[["case_id", "m3t_subset"]], on="case_id")
HEAD["leaky_by_exposure"] = {g: _metrics(d, "leaky_pred") for g, d in _lk.groupby(_lk.m3t_subset == "train")}
HEAD["leaky_by_exposure"] = {("subject_trong_train_M3T" if k else "subject_ngoai_train_M3T"): v
                             for k, v in HEAD["leaky_by_exposure"].items()}
if not HEAD_JSON.exists():
    T.write_once_json(HEAD, HEAD_JSON)
for k in ("all_1229", "s8_test"):
    print(f"head M3T moi | {k}: n={HEAD[k]['n']} QWK {HEAD[k]['qwk']:.3f} acc {HEAD[k]['acc']:.3f} "
          f"MAE {HEAD[k]['mae']:.3f} lech>=2 {HEAD[k]['off_by_2']:.3f}")
for k, v in HEAD["leaky_by_exposure"].items():
    print(f"head RO RI | {k}: n={v['n']} QWK {v['qwk']:.3f} acc {v['acc']:.3f}")

## Ghi chú

- **Một chỗ lệch có chủ ý so với bản gốc:** chọn epoch theo trung bình trượt trailing 5 epoch của val QWK (bản gốc:
  val acc của một epoch, dao động ±0,016). Mọi thứ khác của công thức giữ nguyên.
- **Tái lập:** thứ tự dữ liệu và tăng cường tái lập theo seed từng epoch; trên GPU với `cudnn.benchmark` số học
  **không** bit-đối-bit — giới hạn đã biết. Chạy tiếp bị từ chối khi cấu hình, pool hay torchio khác.
- **Chuyển đổi trượt** → đường `npz_intersection`: S8/S7 phải chạy **mọi** feature set trên cùng giao, giữ nguyên thành
  viên split đã ghim, báo lại n và phân bố KL thực tế.
- **Chưa làm ở đây:** S8/S7 với feature set `m3t_` (bước 7 của kế hoạch), holdout 96 ca (S10).